# 05 RAG Pipeline

## Objective

This notebook combines document retrieval with a language model to answer questions about fraud detection.

## Goals

- Retrieve relevant document chunks
- Augment prompts with retrieved context
- Generate answers with an LLM
- Build a Fraud Detection Knowledge Assistant

## Use Case

Fraud Detection Knowledge Assistant

In [10]:
import faiss
import pandas as pd

from pathlib import Path
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from transformers import pipeline

# PDF Text Extraction Function
def extract_pdf_text(pdf_path):
  reader = PdfReader(pdf_path)
  text = ""
  for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
      text += page_text + "\n"
  return text

# Text Chunking Function
def create_chunks(text, chunk_size=1000):
  chunks = []
  for i in range(0, len(text), chunk_size):
    chunks.append(text[i:i + chunk_size])
  return chunks

# Load Documents
RAW_DATA_DIR = Path("data/raw")
pdf_files = list(RAW_DATA_DIR.glob("*.pdf"))
documents = []
for pdf_file in pdf_files:
  text = extract_pdf_text(pdf_file)
  documents.append({"file_name": pdf_file.name, "text": text})
documents_df = pd.DataFrame(documents)

# Create Chunks
chunk_records = []
for _, row in documents_df.iterrows():
  chunks = create_chunks(row["text"])
  for idx, chunk in enumerate(chunks):
    chunk_records.append({"file_name": row["file_name"], "chunk_id": idx, "chunk_text": chunk})
chunks_df = pd.DataFrame(chunk_records)
print(f"Total chunks: {len(chunks_df)}")

# Load Embedding Model
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded.")

# Generate Embeddings
embeddings = model.encode(chunks_df["chunk_text"].tolist(), show_progress_bar=True)

# Build FAISS Index
embedding_dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dimension)
index.add(embeddings)
print(f"Vectors stored: {index.ntotal}")

# Retriever Function
def retrieve_context(question, k=3):
  query_embedding = model.encode([question])
  distances, indices = index.search(query_embedding, k)
  context = []
  for idx in indices[0]:
    context.append(chunks_df.iloc[idx]["chunk_text"])
  return "\n\n".join(context)

# LLM laden
generator = pipeline("text-generation", model="distilgpt2")
print("LLM loaded.")

# RAG Function
def ask_question(question):
  if chunks_df.empty:
    raise ValueError("No chunks available.")

  context = retrieve_context(question)
  prompt = f"""
  Context:

  {context}

  Question:

  {question}

  Answer:
  """
  answer = generator(prompt, max_new_tokens=200)
  return answer[0]["generated_text"]

question = ("How does the paper detect credit card fraud?")
answer = ask_question(question)
print(answer)


Total chunks: 88


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Vectors stored: 88


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=200) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LLM loaded.

  Context: 
  
  e fraudster assumes a false 
identity with the collusion of the cardholder and the financial institution. In contrast, external fraud involves 
unauthorized access to a credit card to extract money or make transactions through deceptive means1.
Financial fraud detection has emerged as a critical area of research due to the increasing reliance on digital 
financial transactions as well as the growing sophistication of fraud schemes. The rapid expansion of e-commerce, 
online banking, and cashless payment methods has created an urgent need for effective fraud detection systems. 
These systems are used to mitigate financial losses and protect consumers and institutions 2. Fraudulent 
transactions, particularly in credit card payments, often involve unauthorized access through phishing, data 
breaches, and cyber scams, making traditional rule-based detection methods insufficient in handling modern 
fraud schemes3,4.
1Faculty of Computers and Information, Minia

In [11]:
!git add .
!git commit -m "Add RAG pipeline"
!git push

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@256fddc62766.(none)')
Everything up-to-date
